# Downloading a subset from CHAMMI-75

This notebook downloads png images to a folder. Afterwards, we can compute and visualize embeddings from these images.

The code here is a modified version of [download_chammi_small.py](https://github.com/CaicedoLab/CHAMMI-75/blob/f1b066a97e01f6db7e88897f5a631f98e6663d55/aws-tutorials/download_chammi_small.py) which was licensed [MIT](https://github.com/CaicedoLab/CHAMMI-75/blob/main/LICENSE).

It downloads a small fraction of the [CHAMMI-75](https://morgridge.org/research/labs/caicedo/chammi-75/) microscopy images dataset, which is licensed [CC-BY 4.0](https://creativecommons.org/licenses/by/4.0/deed.en).

In [ ]:
import json
import os
import argparse
import requests
import time
import random
from pprint import pprint
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed
from io import BytesIO
import pandas as pd
from urllib.parse import quote

In [ ]:
num_images_to_download = 1000
DOWNLOAD_FOLDER = "./images"
NUM_WORKERS = 8

# Define folder structure
IMAGES_FOLDER = os.path.join(DOWNLOAD_FOLDER, "CHAMMI-75_small")

BASE_URL_TRAIN = "https://chammi-data.s3.amazonaws.com/CHAMMI-75/CHAMMI-75_train"
BASE_URL_GUIDANCE = "https://chammi-data.s3.amazonaws.com/CHAMMI-75/CHAMMI-75_guidance"
METADATA_URL = "https://chammi-data.s3.amazonaws.com/CHAMMI-75/CHAMMI-75_small_metadata.csv"

In [ ]:
def parse_args():
    parser = argparse.ArgumentParser(description="Download CHAMMI-75 dataset images")
    parser.add_argument(
        "--download-folder",
        type=str,
        required=True,
        help="Folder path where images will be downloaded"
    )
    parser.add_argument(
        "--workers",
        type=int,
        default=16,
        help="Number of parallel workers for downloading (default: 16)"
    )
    parser.add_argument(
        "--guidance",
        action="store_true",
        help="If set, also download guidance files (.safetensors format)"
    )
    return parser.parse_args()


def download_file(url, local_path, timeout=30, allow_404=False, max_retries=5):
    """Download a single file from URL to local path with retry logic."""
    os.makedirs(os.path.dirname(local_path), exist_ok=True)
    
    for attempt in range(max_retries):
        try:
            response = requests.get(url, timeout=timeout)
            if allow_404 and response.status_code == 404:
                return True, f"{local_path}: skipped (not found)"
            response.raise_for_status()
            with open(local_path, 'wb') as f:
                f.write(response.content)
            return True, local_path
        except Exception as e:
            if allow_404 and "404" in str(e):
                return True, f"{local_path}: skipped (not found)"
            
            # Don't retry on last attempt
            if attempt == max_retries - 1:
                return False, f"{local_path}: {e}"
            
            # Exponential backoff with jitter
            wait_time = (2 ** attempt) + random.uniform(0, 1)
            time.sleep(wait_time)


def get_missing_images(clean_paths, download_folder):
    """Check which images are missing from the download folder."""
    missing = []
    for path in clean_paths:
        local_path = os.path.join(download_folder, path)
        if not os.path.exists(local_path):
            missing.append(path)
    return missing


def get_missing_guidance(clean_paths, download_folder):
    """Check which safetensor files are missing from the download folder."""
    missing = []
    for path in clean_paths:
        safetensor_path = os.path.splitext(path)[0] + ".safetensors"
        local_path = os.path.join(download_folder, safetensor_path)
        if not os.path.exists(local_path):
            missing.append(path)  # Return original path so download_guidance can convert it
    return missing


def download_image(path, base_url, download_folder):
    """Download a single image."""
    # Pre-encode the path for S3 - it expects %2B, %5B, %5D, etc.
    # quote() with safe='/' keeps forward slashes but encodes +, [, ]
    encoded_path = quote(path, safe='/')
    url = f"{base_url}/{encoded_path}"
    local_path = os.path.join(download_folder, path)
    return download_file(url, local_path)


def download_guidance(path, base_url, download_folder):
    """Download guidance file - convert .png path to .safetensor."""
    # Replace .png extension with .safetensor
    safetensor_path = os.path.splitext(path)[0] + ".safetensors"
    # Pre-encode the path for S3
    encoded_path = quote(safetensor_path, safe='/')
    url = f"{base_url}/{encoded_path}"
    local_path = os.path.join(download_folder, safetensor_path)
    return download_file(url, local_path, allow_404=True)

In [4]:
# Create the download folders
os.makedirs(DOWNLOAD_FOLDER, exist_ok=True)
os.makedirs(IMAGES_FOLDER, exist_ok=True)

# Download metadata CSV to main folder
metadata_local_path = os.path.join(DOWNLOAD_FOLDER, "CHAMMI-75_small_metadata.csv")
if not os.path.exists(metadata_local_path):
    print("Downloading metadata CSV...")
    success, result = download_file(METADATA_URL, metadata_local_path)
    if success:
        print(f"Metadata downloaded to: {metadata_local_path}")
    else:
        print(f"Failed to download metadata: {result}")
        

# Load the metadata
print("Opening metadata CSV...")
df = pd.read_csv(metadata_local_path)

# Get the clean paths
clean_paths = df["storage.path"].str.split("/").apply(lambda x: "/".join(x[1:])).tolist()

# Test first with a sample URL
sample_path = clean_paths[0]
sample_encoded_path = quote(sample_path, safe='/')
sample_url = f"{BASE_URL_TRAIN}/{sample_encoded_path}"
response = requests.get(sample_url)

if response.status_code != 200:
    print(f"URL structure is wrong - check the path")
    print(f"Sample URL: {sample_url}")
    print(f"Status code: {response.status_code}")
    

print(f"Sample URL test passed: {response.status_code}")

# Download images to CHAMMI-75_small folder
# First check which images are missing
print(f"\nChecking for missing images in {IMAGES_FOLDER}...")
missing_images = get_missing_images(clean_paths, IMAGES_FOLDER)

print(type(missing_images))
print(len(missing_images))


C:\Users\rober\AppData\Local\Temp\ipykernel_22276\237367849.py:18: DtypeWarning: Columns (1,2,13,14,17,19) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(metadata_local_path)


Sample URL test passed: 200

Checking for missing images in ./images\CHAMMI-75_small...
<class 'list'>
1679740


Select n random images.

In [5]:
missing_images = random.sample(missing_images, num_images_to_download)

missing_images[:10]

['hpa0023/1164/1164_D5_2_10_cell_image_channel-1.png',
 'idr0056/idr0056-plate_8B-converted/015016025_series-0_z-0_t-0_channel-0.png',
 'hpa0023/2142/2142_C8_2_2_cell_image_channel-1.png',
 'wtc0001/PIC/FOV/e63ab3ee_3500000959_100X_20170609_1-Scene-1-P1-E04_z15-ch0.png',
 'nidr0032/nidr0032-plate_2A-converted/K29_s2_w3.png',
 'jump0001/cpg0016-jump/source_13/images/20220914_Run1/images_compressed/CP-CC9-R1-11/CP-CC9-R1-11_K12_T0001F006L01A01Z01C01.png',
 'idr0002/idr0002-plate_3A-converted/C4--W00028--P00001--Z00000--T00333--Cy3_series-0_z-0_t-0_channel-0.png',
 'idr0094/idr0094-plate_83B-converted/2020-05-06T110258+0200[6754]_2020-05-06T110258+0200[6754]_004004-6-001001001_series-0_z-0_t-0_channel-0.png',
 'jump0001/cpg0016-jump/source_13/images/20221009_Run2/images_compressed/CP-CC9-R2-13/CP-CC9-R2-13_M06_T0001F007L01A03Z01C04.png',
 'idr0009/idr0009-plate_640A-converted/--W00102--P00001--Z00000--T00000--vsvg-cfp_series-0_z-0_t-0_channel-0.png']

Check if they are all png files.

In [6]:
for i in missing_images:
    if not i.endswith(".png"):
        print(i)

In [7]:
print(f"Downloading selected images with {NUM_WORKERS} workers...")
failed_images = []

with ThreadPoolExecutor(max_workers=NUM_WORKERS) as executor:
    futures = {
        executor.submit(download_image, path, BASE_URL_TRAIN, IMAGES_FOLDER): path 
        for path in missing_images
    }
    
    for future in tqdm(as_completed(futures), total=len(missing_images), desc="Images"):
        success, result = future.result()
        if not success:
            failed_images.append(result)

if failed_images:
    print(f"\nFailed image downloads ({len(failed_images)}):")
    for f in failed_images[:10]:
        print(f"  {f}")
else:
    print(f"\nAll {len(missing_images)} missing images downloaded successfully!")


print("\nDownload complete!")

Images: 100%|██████████████████████████████████████████████████████████████████████| 1000/1000 [02:55<00:00,  5.70it/s]


All 1000 missing images downloaded successfully!

Download complete!
